## 1. Setup & Installation

In [ ]:
# Check GPU
!nvidia-smi

# Check if we're on Colab
import os
IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
print(f"Running on Colab: {IN_COLAB}")

In [ ]:
# Clone repository (if on Colab)
import os

REPO_URL = "https://github.com/Pkansagra-hub/Family_osModernBERT.git"
REPO_DIR = "Modeling_studio"

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        print("📥 Cloning repository...")
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print("📂 Repository already exists, pulling latest...")
        !cd {REPO_DIR} && git pull

    os.chdir(REPO_DIR)
    print(f"📁 Working directory: {os.getcwd()}")
else:
    # Local development - assume we're in the repo root
    print(f"📁 Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("📦 Installing dependencies...")
!pip install -q -e .
!pip install -q flash-attn --no-build-isolation 2>/dev/null || echo "Flash Attention not installed (optional)"
!pip install -q wandb tensorboard

print("✅ Dependencies installed!")

In [ ]:
# Mount Google Drive (for data and checkpoints persistence)
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Create symlinks for data and outputs
    DRIVE_BASE = "/content/drive/MyDrive/FamilyOS_ModernBERT"
    !mkdir -p "{DRIVE_BASE}/data"
    !mkdir -p "{DRIVE_BASE}/outputs"
    !mkdir -p "{DRIVE_BASE}/checkpoints"

    # Symlink outputs to Drive for persistence
    !rm -rf outputs checkpoints 2>/dev/null
    !ln -s "{DRIVE_BASE}/outputs" outputs
    !ln -s "{DRIVE_BASE}/checkpoints" checkpoints

    print(f"📂 Data/outputs will be saved to: {DRIVE_BASE}")

## 2. Configuration

In [ ]:
# Training Configuration
import torch

# ============================================
# MODIFY THESE SETTINGS AS NEEDED
# ============================================

# Stage A Settings
STAGE_A_EPOCHS = 4          # Recommended: 3-4 for A100
STAGE_A_BATCH_SIZE = 128    # A100-80GB: 192, A100-40GB: 96, T4: 32

# Stage B Settings
STAGE_B_EPOCHS = 3          # Recommended: 2-3 for v3 prep
STAGE_B_BATCH_SIZE = 64     # A100-80GB: 64, A100-40GB: 32, T4: 16

# Detect GPU and auto-adjust batch sizes
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"🎮 GPU: {gpu_name} ({gpu_memory:.1f} GB)")

    if "A100" in gpu_name and gpu_memory > 70:
        STAGE_A_BATCH_SIZE = 192
        STAGE_B_BATCH_SIZE = 64
        print("   → Using A100-80GB settings")
    elif "A100" in gpu_name:
        STAGE_A_BATCH_SIZE = 96
        STAGE_B_BATCH_SIZE = 32
        print("   → Using A100-40GB settings")
    elif "T4" in gpu_name:
        STAGE_A_BATCH_SIZE = 32
        STAGE_B_BATCH_SIZE = 16
        print("   → Using T4 settings")
    elif "V100" in gpu_name:
        STAGE_A_BATCH_SIZE = 48
        STAGE_B_BATCH_SIZE = 24
        print("   → Using V100 settings")
else:
    print("⚠️ No GPU detected! Training will be very slow.")

print(f"\n📊 Stage A: {STAGE_A_EPOCHS} epochs, batch_size={STAGE_A_BATCH_SIZE}")
print(f"📊 Stage B: {STAGE_B_EPOCHS} epochs, batch_size={STAGE_B_BATCH_SIZE}")

In [ ]:
# Optional: Login to Weights & Biases for experiment tracking
# Uncomment below if you want W&B logging

# import wandb
# wandb.login()
# os.environ["WANDB_PROJECT"] = "familyos-modernbert"

print("ℹ️ W&B logging disabled. Uncomment above to enable.")

## 3. Verify Data

In [ ]:
import os
from pathlib import Path

def check_data_exists():
    """Verify required data directories exist."""
    required_paths = {
        "Stage A - NER": "data/public/civil_comments_curated",
        "Stage A - Emotions": "data/familyos/emotions/silver",
        "Stage A - Temporal": "data/familyos/temporal/silver",
    }

    optional_paths = {
        "Stage B - Unified Synthetic": "data/familyos/unified/output_synthetic",
        "Stage B - Unified": "data/familyos/unified/output",
    }

    print("📂 Checking required data...")
    all_ok = True

    for name, path in required_paths.items():
        exists = os.path.exists(path)
        status = "✅" if exists else "❌"
        print(f"   {status} {name}: {path}")
        if not exists:
            all_ok = False

    print("\n📂 Checking optional data (Stage B)...")
    stage_b_ok = False
    for name, path in optional_paths.items():
        exists = os.path.exists(path)
        status = "✅" if exists else "⚠️"
        print(f"   {status} {name}: {path}")
        if exists:
            stage_b_ok = True

    return all_ok, stage_b_ok

stage_a_ready, stage_b_ready = check_data_exists()

if not stage_a_ready:
    print("\n❌ Stage A data missing! Please upload data or mount Google Drive.")
elif not stage_b_ready:
    print("\n⚠️ Stage B data missing - will skip Stage B training.")
else:
    print("\n✅ All data ready!")

## 4. Stage A: Train on Public Datasets

In [ ]:
%%time

import os

print("="*60)
print("🚀 STAGE A: Training on Public Datasets")
print("="*60)
print(f"   Epochs: {STAGE_A_EPOCHS}")
print(f"   Batch size: {STAGE_A_BATCH_SIZE}")
print(f"   Output: outputs/modernbert-multitask-v0-stage-a-fast")
print("="*60)

# Use shell command with ! to stream output in real-time
!python scripts/train_stage_a.py \
    --config configs/training/multitask/stage_a_a100_fast.yaml \
    training.num_train_epochs={STAGE_A_EPOCHS} \
    training.per_device_train_batch_size={STAGE_A_BATCH_SIZE}

# Check if training succeeded
if os.path.exists("outputs/modernbert-multitask-v0-stage-a-fast/model.safetensors"):
    print("\n" + "="*60)
    print("✅ STAGE A COMPLETED SUCCESSFULLY!")
    print("="*60)
else:
    print("\n" + "="*60)
    print("❌ STAGE A FAILED! Check logs above.")
    print("="*60)

In [ ]:
# Verify Stage A output
import os
import json

stage_a_output = "outputs/modernbert-multitask-v0-stage-a-fast"

if os.path.exists(stage_a_output):
    print(f"📁 Stage A output: {stage_a_output}")
    print("   Files:")
    for f in os.listdir(stage_a_output):
        size = os.path.getsize(os.path.join(stage_a_output, f)) / 1e6
        print(f"      {f} ({size:.1f} MB)")

    # Load eval results
    eval_path = os.path.join(stage_a_output, "eval_results.json")
    if os.path.exists(eval_path):
        with open(eval_path) as f:
            results = json.load(f)
        print("\n📊 Stage A Eval Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"      {k}: {v:.4f}")
else:
    print(f"❌ Stage A output not found at {stage_a_output}")

## 5. Stage B: Fine-tune on FamilyOS Data

In [ ]:
%%time

import os

# Check if Stage B data exists
stage_b_data_exists = (
    os.path.exists("data/familyos/unified/output_synthetic") or
    os.path.exists("data/familyos/unified/output")
)

if not stage_b_data_exists:
    print("⚠️ Stage B data not found. Skipping Stage B training.")
    print("   To run Stage B, ensure FamilyOS unified data exists at:")
    print("   - data/familyos/unified/output_synthetic")
    print("   - data/familyos/unified/output")
else:
    print("="*60)
    print("🚀 STAGE B: Fine-tuning on FamilyOS Data")
    print("="*60)
    print(f"   Base model: outputs/modernbert-multitask-v0-stage-a-fast")
    print(f"   Epochs: {STAGE_B_EPOCHS}")
    print(f"   Batch size: {STAGE_B_BATCH_SIZE}")
    print(f"   Output: outputs/modernbert-v2-for-v3-transfer")
    print("="*60)

    # Use shell command with ! to stream output in real-time
    !python scripts/train_stage_b.py \
        --config configs/training/multitask/stage_b_for_v3_prep.yaml \
        training.num_train_epochs={STAGE_B_EPOCHS} \
        training.per_device_train_batch_size={STAGE_B_BATCH_SIZE}

    # Check if training succeeded
    if os.path.exists("outputs/modernbert-v2-for-v3-transfer/model.safetensors"):
        print("\n" + "="*60)
        print("✅ STAGE B COMPLETED SUCCESSFULLY!")
        print("="*60)
    else:
        print("\n" + "="*60)
        print("❌ STAGE B FAILED! Check logs above.")
        print("="*60)

In [ ]:
# Verify Stage B output
import os
import json

stage_b_output = "outputs/modernbert-v2-for-v3-transfer"

if os.path.exists(stage_b_output):
    print(f"📁 Stage B output: {stage_b_output}")
    print("   Files:")
    for f in os.listdir(stage_b_output):
        size = os.path.getsize(os.path.join(stage_b_output, f)) / 1e6
        print(f"      {f} ({size:.1f} MB)")

    # Load eval results
    eval_path = os.path.join(stage_b_output, "eval_results.json")
    if os.path.exists(eval_path):
        with open(eval_path) as f:
            results = json.load(f)
        print("\n📊 Stage B Eval Results:")
        for k, v in sorted(results.items()):
            if isinstance(v, float):
                print(f"      {k}: {v:.4f}")
else:
    print(f"ℹ️ Stage B output not found (may have been skipped)")

## 6. Summary & Next Steps

In [ ]:
import os
from datetime import datetime

print("="*60)
print("📋 TRAINING PIPELINE SUMMARY")
print("="*60)
print(f"   Completed at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Check outputs
stage_a_ok = os.path.exists("outputs/modernbert-multitask-v0-stage-a-fast/model.safetensors")
stage_b_ok = os.path.exists("outputs/modernbert-v2-for-v3-transfer/model.safetensors")

print(f"\n   Stage A: {'✅ Complete' if stage_a_ok else '❌ Failed/Skipped'}")
print(f"   Stage B: {'✅ Complete' if stage_b_ok else '⚠️ Skipped (no FamilyOS data)'}")

print("\n" + "="*60)
print("📦 OUTPUT CHECKPOINTS")
print("="*60)

if stage_a_ok:
    print("   Stage A: outputs/modernbert-multitask-v0-stage-a-fast")
    print("            → Generic multi-task encoder (7 heads)")

if stage_b_ok:
    print("   Stage B: outputs/modernbert-v2-for-v3-transfer")
    print("            → FamilyOS-tuned encoder for v3 weight transfer")
    print("            → Layers 15-20 trained on family context")

print("\n" + "="*60)
print("🔜 NEXT STEPS")
print("="*60)
print("   1. Download checkpoints from Google Drive")
print("   2. Run v3 initialization script to:")
print("      - Copy v2 layers 1-22 → v3 layers 1-22")
print("      - Clone v2 layers 15-20 → v3 layers 23-28")
print("      - Initialize new v3 heads + hub tokens")
print("   3. Train v3 on FamilyOS data with new architecture")
print("="*60)

In [ ]:
# Optional: Copy outputs to a specific Google Drive location
if IN_COLAB:
    import shutil
    from datetime import datetime

    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    backup_dir = f"/content/drive/MyDrive/FamilyOS_ModernBERT/runs/{timestamp}"

    print(f"📦 Creating backup at: {backup_dir}")
    os.makedirs(backup_dir, exist_ok=True)

    # Copy Stage A
    if os.path.exists("outputs/modernbert-multitask-v0-stage-a-fast"):
        shutil.copytree(
            "outputs/modernbert-multitask-v0-stage-a-fast",
            f"{backup_dir}/stage_a",
            dirs_exist_ok=True
        )
        print("   ✅ Stage A backed up")

    # Copy Stage B
    if os.path.exists("outputs/modernbert-v2-for-v3-transfer"):
        shutil.copytree(
            "outputs/modernbert-v2-for-v3-transfer",
            f"{backup_dir}/stage_b",
            dirs_exist_ok=True
        )
        print("   ✅ Stage B backed up")

    print(f"\n✅ Backup complete: {backup_dir}")